# Market Size Analysis on Electric Vehicles Industries
**Python Project | Prepared by Nikhilkumar Ghatage**

## Project overview
This notebook analyses the supplied Electric Vehicle Population dataset to understand EV registrations, model-year patterns, geographic distribution, manufacturer/model popularity, EV technology types, electric range, price-range relationships, and an illustrative trend projection.

**Important scope note:** the source dataset is a population/registration snapshot, not a complete measure of all EV sales or market revenue. Results should be interpreted as counts and characteristics of records in this file, not as the entire US EV market. The accompanying project brief focuses especially on Washington State (WA).

## Objectives
1. Assess historical registrations by model year.
2. Explore geographic distribution by state, county and city.
3. Identify popular makes, models and EV types.
4. Summarise range and MSRP data and assess their relationship.
5. Demonstrate a simple, validated trend model and document limitations.
6. Explore geographic hotspots using clustering of supplied coordinates.

## 1. Setup and load data
Place `Electric_Vehicle_Population_Data_Excel_sheet.xlsx` in the same folder as this notebook. The notebook uses pandas for data handling, Matplotlib/Seaborn for visualisation, and scikit-learn for regression and clustering.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.cluster import KMeans

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid")

candidate_paths = [
    Path("Electric_Vehicle_Population_Data_Excel_sheet.xlsx"),
    Path("/mnt/data/Electric_Vehicle_Population_Data_Excel_sheet.xlsx")
]
DATA_PATH = next((p for p in candidate_paths if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Place the supplied Excel dataset beside this notebook.")
raw_df = pd.read_excel(DATA_PATH)
df = raw_df.copy()
print(f"Loaded {len(df):,} rows and {df.shape[1]} columns.")
display(df.head())
display(df.info())

## 2. Data cleaning and preparation
Cleaning is intentionally conservative. Text fields are trimmed, numeric columns are coerced to numeric, and exact duplicate rows are removed. Zero electric-range and MSRP values are retained in the main dataset because they may represent unavailable/unreported values rather than true zeroes; positive values are used for range/price descriptive summaries.

In [ ]:
original_rows = len(df)
df.columns = [c.strip() for c in df.columns]
numeric_cols = ["Postal Code", "Model Year", "Electric Range", "Base MSRP",
                "Legislative District", "DOL Vehicle ID", "2020 Census Tract"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
for col in df.select_dtypes(include=["object"]).columns:
    df[col] = df[col].astype("string").str.strip()
    df[col] = df[col].replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
duplicate_rows = int(df.duplicated().sum())
df = df.drop_duplicates().copy()

missing_summary = df.isna().sum().sort_values(ascending=False).rename("missing_count").to_frame()
missing_summary["missing_pct"] = (missing_summary["missing_count"] / len(df) * 100).round(3)
print(f"Rows before cleaning: {original_rows:,}")
print(f"Exact duplicate rows removed: {duplicate_rows:,}")
print(f"Rows after cleaning: {len(df):,}")
display(missing_summary)

## 3. Descriptive analysis
The summaries below describe positive range and MSRP values separately so zero placeholders do not distort the averages.

In [ ]:
range_positive = df.loc[df["Electric Range"].notna() & (df["Electric Range"] > 0), "Electric Range"]
msrp_positive = df.loc[df["Base MSRP"].notna() & (df["Base MSRP"] > 0), "Base MSRP"]
print("Positive electric range summary (miles):")
display(range_positive.describe().to_frame("Electric Range"))
print("Positive Base MSRP summary (USD):")
display(msrp_positive.describe().to_frame("Base MSRP"))
print("EV type counts:")
display(df["Electric Vehicle Type"].value_counts(dropna=False).to_frame("records"))

## 4. Historical model-year trend
A model year is not the same as the date a vehicle was registered. The trend shows the number of rows by vehicle model year in this dataset; it should not be interpreted as annual sales or registrations occurring in that calendar year.

In [ ]:
annual = (df.dropna(subset=["Model Year"])
          .groupby("Model Year").size()
          .rename("Records").reset_index()
          .sort_values("Model Year"))
plt.figure(figsize=(10, 5))
sns.lineplot(data=annual, x="Model Year", y="Records", marker="o")
plt.title("EV Records by Model Year")
plt.xlabel("Model year"); plt.ylabel("Number of records")
plt.tight_layout(); plt.show()
display(annual)

## 5. Geographic distribution
The source brief highlights Washington State. The dataset's state counts are checked first; county and city rankings are then calculated. These are record counts, not per-capita adoption rates, because population denominators are not provided.

In [ ]:
display(df["State"].value_counts(dropna=False).rename("records").to_frame())
top_counties = df["County"].value_counts().head(10)
top_cities = df["City"].value_counts().head(10)
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
top_counties.sort_values().plot(kind="barh", ax=axes[0])
axes[0].set_title("Top 10 Counties"); axes[0].set_xlabel("Records")
top_cities.sort_values().plot(kind="barh", ax=axes[1])
axes[1].set_title("Top 10 Cities"); axes[1].set_xlabel("Records")
plt.tight_layout(); plt.show()
display(top_counties.rename("records").to_frame())
display(top_cities.rename("records").to_frame())

## 6. Make, model and EV-type popularity

In [ ]:
top_makes = df["Make"].value_counts().head(10)
top_models = df["Model"].value_counts().head(10)
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
top_makes.sort_values().plot(kind="barh", ax=axes[0])
axes[0].set_title("Top 10 Makes"); axes[0].set_xlabel("Records")
top_models.sort_values().plot(kind="barh", ax=axes[1])
axes[1].set_title("Top 10 Models"); axes[1].set_xlabel("Records")
plt.tight_layout(); plt.show()
display(df["Electric Vehicle Type"].value_counts().rename("records").to_frame())
display(df["Clean Alternative Fuel Vehicle (CAFV) Eligibility"].value_counts(dropna=False).rename("records").to_frame())

## 7. Electric range and MSRP
Range and MSRP can be missing, zero, or not representative of current transaction prices. For correlation, this notebook uses rows where both fields are positive. Correlation is descriptive and does not establish causation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(range_positive, bins=35, ax=axes[0])
axes[0].set_title("Positive Electric Range"); axes[0].set_xlabel("Range (miles)")
sns.histplot(msrp_positive, bins=35, ax=axes[1])
axes[1].set_title("Positive Base MSRP"); axes[1].set_xlabel("Base MSRP (USD)")
plt.tight_layout(); plt.show()

corr_data = df.loc[(df["Electric Range"] > 0) & (df["Base MSRP"] > 0),
                   ["Electric Range", "Base MSRP"]].dropna()
corr = corr_data.corr().iloc[0, 1] if len(corr_data) > 1 else np.nan
print(f"Pearson correlation between positive Electric Range and positive Base MSRP: {corr:.3f}")
plt.figure(figsize=(8, 5))
sns.scatterplot(data=corr_data.sample(min(5000, len(corr_data)), random_state=42),
                x="Electric Range", y="Base MSRP", alpha=.5, s=18)
plt.title("Electric Range vs Base MSRP (sample up to 5,000 rows)")
plt.tight_layout(); plt.show()

## 8. Geographic coordinate extraction and clustering
Coordinates are parsed from `Vehicle Location` values formatted as `POINT (longitude latitude)`. K-means clustering is used as an exploratory hotspot demonstration only. It is not a formal spatial-statistics model, and cluster numbers have no inherent meaning.

In [ ]:
coords = df["Vehicle Location"].astype("string").str.extract(
    r"POINT\s*\(\s*([-0-9.]+)\s+([-0-9.]+)\s*\)"
)
df["Longitude"] = pd.to_numeric(coords[0], errors="coerce")
df["Latitude"] = pd.to_numeric(coords[1], errors="coerce")
geo = df.dropna(subset=["Longitude", "Latitude"]).copy()
print(f"Rows with usable coordinates: {len(geo):,} of {len(df):,}")
if len(geo) >= 5:
    sample_geo = geo[["Longitude", "Latitude"]].sample(min(10000, len(geo)), random_state=42)
    kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
    sample_geo["Cluster"] = kmeans.fit_predict(sample_geo[["Longitude", "Latitude"]])
    plt.figure(figsize=(8, 6))
    sns.scatterplot(data=sample_geo, x="Longitude", y="Latitude", hue="Cluster",
                    palette="tab10", s=10, alpha=.5, legend="full")
    plt.title("Exploratory K-means Clusters of EV Coordinates")
    plt.tight_layout(); plt.show()
    display(sample_geo["Cluster"].value_counts().sort_index().rename("sample_records").to_frame())
else:
    print("Not enough valid coordinates for clustering.")

## 9. Illustrative forecasting
This section aggregates records by model year and uses a simple linear regression with the latest 20% (at least three points) held out for evaluation when enough years are available. Because the dataset is a snapshot and model year differs from registration year, this is only an illustrative trend projection. A poor or negative test R² indicates the linear trend is not reliable for prediction.

In [ ]:
annual = annual[annual["Model Year"] >= 2010].copy()
if len(annual) >= 6:
    test_n = max(3, int(np.ceil(len(annual) * 0.2)))
    train, test = annual.iloc[:-test_n], annual.iloc[-test_n:]
    model = LinearRegression().fit(train[["Model Year"]], train["Records"])
    test_pred = model.predict(test[["Model Year"]])
    mae = mean_absolute_error(test["Records"], test_pred)
    rmse = mean_squared_error(test["Records"], test_pred) ** 0.5
    r2 = r2_score(test["Records"], test_pred) if len(test) > 1 else np.nan
    print(f"Holdout MAE: {mae:,.1f}; RMSE: {rmse:,.1f}; R²: {r2:.3f}")
    final_model = LinearRegression().fit(annual[["Model Year"]], annual["Records"])
    future_years = np.arange(int(annual["Model Year"].max()) + 1,
                             int(annual["Model Year"].max()) + 6)
    future_pred = np.maximum(0, final_model.predict(pd.DataFrame({"Model Year": future_years})))
    forecast = pd.DataFrame({"Model Year": future_years,
                             "Illustrative projected records": np.round(future_pred).astype(int)})
    display(forecast)
    plt.figure(figsize=(10, 5))
    plt.plot(annual["Model Year"], annual["Records"], marker="o", label="Historical records")
    plt.plot(future_years, future_pred, marker="o", linestyle="--", label="Linear projection")
    plt.title("Illustrative Model-Year Trend Projection")
    plt.xlabel("Model year"); plt.ylabel("Number of records"); plt.legend()
    plt.tight_layout(); plt.show()
else:
    print("Not enough annual observations for a holdout forecast.")

## 10. Summary and recommendations
- Use county/city counts to identify where the supplied records are concentrated, but do not label these as per-capita adoption rates without population denominators.
- Interpret range and MSRP summaries using positive values and clearly disclose that zero values were excluded from those specific summaries.
- Treat correlations as associations, not evidence that range causes higher prices.
- Treat the linear projection as illustrative only. Model year is not registration year and the validation results should guide whether the model is useful.
- For more reliable market sizing, obtain a time-stamped registration series, population denominators, EV sales/stock coverage information, and consistent vehicle price data.